# <div style="text-align: center; background-color: #0C6A86; font-family:newtimeroman; color: white; padding: 14px; line-height: 1;border-radius:20px">📊**Capastone**</div>


# imports 

#basic imports 
import pandas as pd 
from sklearn.ensemble import GradientBoostingClassifier
import numpy as np 
import matplotlib.pyplot as plt 


data = pd.read_csv('data/train.csv')


# data = data.set_index('observation_id')
data.head()

import matplotlib.pyplot as plt

non_null_values = data.notnull().sum()

plt.figure(figsize=(10, 6))

# Adjust the height and padding between the bars
height = 0.7
padding = 1

y = range(len(non_null_values))
bars = plt.barh([i * (height + padding) for i in y], non_null_values.values, height=height)

plt.yticks([i * (height + padding) for i in y], non_null_values.index, fontsize=12)
plt.xticks(fontsize=12)
plt.ylabel('Features', fontsize=15)
plt.xlabel('Count/Frequency', fontsize=15)

# Add the number of non-null values on the right side of each bar
for bar in bars:
    width = bar.get_width()
    plt.text(
        width + max(non_null_values.values) * 0.02,  # Adjust the multiplier for desired label distance
        bar.get_y() + bar.get_height() / 2,
        str(width),
        va='center',
        fontsize=12
    )

plt.tight_layout()

# Adjust the plot limits and add padding
plt.ylim(y[0] * (height + padding) - padding, y[-1] * (height + padding) + height + padding)

plt.box(False)
plt.savefig('horizontal_bar_chart.png')
plt.show()


# Succes searching 

    # which has positive outcome(any value except A no further action) and True Outcome linked 
data[(data['Outcome']!='A no further action disposal') & (data['Outcome linked to object of search']==True) ].shape

data['Outcome map'] = data['Outcome'].apply(lambda x: True if x != 'A no further action disposal' else False)

# stations that has most part of policing operation and the ones that its part of policing operation is false and the ones that have nans

def getting_counts(data,col_index , col_columns , normalize = False ):
    cop = data.copy()
    cop[col_index] =  cop[col_index].astype(str)
    group = cop.groupby([col_index, 
                col_columns]).count()['observation_id'].unstack()
    
    stations_that_all_nulls = [x for x in cop[col_index].unique() if x not in group.index]
    second  =cop[data[col_columns].isnull()][col_index].value_counts()
    group['missing'] = np.nan
    
    for id in stations_that_all_nulls:
        group.loc[id] = np.nan
    group.loc[second.index,'missing'] =  second.values
#     return group 
    if normalize : 
        return round(group.div(group.sum(axis=1 ), axis=0),3).fillna(0)
    else : 
        return round(group,3).fillna(0).astype(int)
# pd.crosstab(data['station'],data['Outcome linked to object of search'],normalize='index')

import matplotlib.pyplot as plt

def plot_chart(data):
    # Generate a color palette with unique colors for each category
    unique_categories = data.columns
    num_categories = len(unique_categories)
    colors = plt.cm.get_cmap('Set3', num_categories)

    fig, ax = plt.subplots(figsize=(8, 4))
    data.T.plot(kind='barh', stacked=True, ax=ax, color=colors(range(num_categories)))
    ax.set_ylabel("Removal of more than just outer clothing",fontsize= 12)
    ax.set_xlabel("Count")
    ax.set_title("Distribution of Observations by Removal and Female")
    ax.legend(loc='upper right')
    ax.grid(False)  # Remove grid lines
    ax.spines['top'].set_visible(False)  # Remove top border
    ax.spines['right'].set_visible(False)  # Remove right border
    ax.spines['bottom'].set_visible(False)  # Remove bottom border
    ax.spines['left'].set_visible(False)  # Remove left border
    ax.set_frame_on(False)  # Remove plot frame

    # Add value labels to each bar
    for container in ax.containers:
        ax.bar_label(container, label_type='edge', fontsize=14, padding=3)

    plt.show()


# Assuming 'data' is the input dataframe



temp = getting_counts(data,'station','Part of a policing operation')

temp = getting_counts(data,'station','Outcome linked to object of search',normalize=True)
temp[temp['missing']>.9]

temp = getting_counts(data,'station','Removal of more than just outer clothing',normalize=True)
temp[temp['missing']>.9]

# doing getting counts and cross tab on target columns 

# this one for all the compination
pd.crosstab(data['Outcome map'],data['Outcome linked to object of search'],normalize='all')

# this one for each raw 
pd.crosstab(data['Outcome map'],data['Outcome linked to object of search'],normalize='index')

#include nan into consideration
df = getting_counts(data,'Outcome map' , "Outcome linked to object of search",normalize = True)
heading_properties = [('font-size', '18px')]
cell_properties = [('font-size', '16px'), ('text-align', 'left')]

dfstyle = [
    dict(selector="th", props=heading_properties),
    dict(selector="td", props=cell_properties)
]

styled_df = df.style.set_table_styles(dfstyle)

# Remove the "000" formatting from the values
styled_df = styled_df.format("{:.3f}")

styled_df

# removal of stations that don't have outcome linked and removal 

    # stations that have null in outcome linked 
# values  = list(data[data['Removal of more than just outer clothing'].isnull()]['station'].value_counts().sort_index().index)
# first = data[data['Removal of more than just outer clothing'].isnull()]['station'].value_counts().sort_index()
# print(first)
# print('------------------------------')
# print('------------------------------')
# print(data[data['station'].isin(values)]['station'].value_counts().sort_index())
# print('------------------------------')
# print('------------------------------')
# print(data[data['station'].isin(values)]['station'].value_counts().sort_index()-first)
# stations_with_complete_nans = data[data['station'].isin(values)]['station'].value_counts().sort_index()-first
# stations_with_complete_nans = stations_with_complete_nans[stations_with_complete_nans==0].index
# stations_with_complete_nans
    # stations of complete nans for both removal and outcome linked 
# stations_with_complete_nans_outcome_linked =  ['humberside', 'lancashire', 'metropolitan']
# stations_with_complete_nans_Removal = ['cleveland', 'lancashire', 'metropolitan', 'north-yorkshire', 'surrey']
stations_with_complete_nans_intersaction = ['lancashire', 'metropolitan']    
stations_with_complete_nans = ['humberside','cleveland', 'lancashire', 'metropolitan', 'north-yorkshire', 'surrey']
# print('------------------------------')

data = data[~data['station'].isin(stations_with_complete_nans_intersaction)]
data.shape

# getting the distribution of the stations after removal of the last cell 

getting_counts(data,'station','Removal of more than just outer clothing')

# data where it was vehicle search and there is removal of clothes and that should not be available but they say i should be possible but let it now and consider it in progress

    # data where it was vehicle search and there is removal of clothes and that sholuld not be available 
data.loc[(data['Type']=='Vehicle search') & (data['Removal of more than just outer clothing']==True ),'Removal of more than just outer clothing'] = False
    # data type of person and vehicle and its object of search values 
# data[data['Type']=='Person and Vehicle search']['Object of search'].unique()

# filling nulls for removal and linked

    # missing values before filling 
print('data before filling nulls ')
data.isnull().sum()

getting_counts(data,'Outcome map' , "Outcome linked to object of search")

    # making of new data after filling both removal and linked 
data_filling_nans = data.copy()
data_filling_nans.loc[:,'Outcome linked to object of search'].fillna(False,inplace = True)
    #Trying another way of filling nans it wasn't best
# data_filling_nans.loc[data_filling_nans['Outcome map'] == False, 'Outcome linked to object of search'] = data_filling_nans.loc[data_filling_nans['Outcome map'] == False, 'Outcome linked to object of search'].fillna(False)
# data_filling_nans.loc[data_filling_nans['Outcome map']==True,'Outcome linked to object of search']= data_filling_nans.loc[data_filling_nans['Outcome map']==True,'Outcome linked to object of search'].fillna(True)
# data_filling_nans.loc[data_filling_nans['Type']!='Vehicle search','Removal of more than just outer clothing'] = data_filling_nans.loc[data_filling_nans['Type']!='Vehicle search','Removal of more than just outer clothing'].fillna(value= False )
data_filling_nans.loc[:,'Removal of more than just outer clothing'].fillna(value= False ,inplace = True)
print('Data after filling nulls')
data_filling_nans.isnull().sum()

# objects that been found but get negative outcome and neglect the possible values that have premissions

getting_counts(data_filling_nans,'Outcome map' , "Outcome linked to object of search")

pd.crosstab(data_filling_nans['Outcome map'],data_filling_nans['Outcome linked to object of search'])

possible_having_with_premmision_paper = ['Controlled drugs','Fireworks','Game or poaching equipment','Firearms','Goods on which duty has not been paid etc.','Crossbows','Seals or hunting equipment']
    # objects that been found but get negative outcome
objects = list(data_filling_nans.loc[(data_filling_nans['Outcome']=='A no further action disposal')& (data_filling_nans['Outcome linked to object of search'] ==True),:]['Object of search'].value_counts().index)
for x in possible_having_with_premmision_paper :
    objects.remove(x)
data_filling_nans.loc[(data_filling_nans['Object of search'].isin(objects))&(data_filling_nans['Outcome']=='A no further action disposal')& (data_filling_nans['Outcome linked to object of search'] ==True),'Outcome linked to object of search'] = False

getting_counts(data_filling_nans,'Outcome map' , "Outcome linked to object of search")

pd.crosstab(data_filling_nans['Outcome map'],data_filling_nans['Outcome linked to object of search'])

# removing the age range under 10 that have type contain vechile search and the outcome linked to object of search is True 

values = data_filling_nans.loc[(data_filling_nans['Age range'].isin(['under 10']))&(data_filling_nans['Type'].isin(['Vehicle search','Person and Vehicle search']))& (data_filling_nans['Outcome linked to object of search']==True),:].index
data_filling_nans.drop(index=list(values),inplace=True)

# check how many values was part of policing operation and have true outcome

# data_filling_nans.loc[data_filling_nans['Part of a policing operation']==True]['Outcome linked to object of search'].value_counts()
getting_counts(data_filling_nans,'Part of a policing operation' , "Outcome linked to object of search")

# check value counts for data that have true outcome and outcome linked is false and this could be possible outcome but having something else not the object of search

data_filling_nans.loc[(data_filling_nans['Outcome']!='A no further action disposal')& (data_filling_nans['Outcome linked to object of search'] == False)].head(2)

# something to try is to try to fill the values which is nulls in part of policing operations as False or to get another way 

data_filling_nans['Part of a policing operation'].fillna(False,inplace = True)

    # Checking nulls after filling part of policing 
data_filling_nans.isnull().sum()

# lower casing 

lowering = ['station','Type','Age range','Gender','Officer-defined ethnicity','Legislation','Object of search']
data_filling_nans[lowering] = data_filling_nans[lowering].applymap(lambda x: str(x).lower() if pd.notna(x) else x)

data_filling_nans.isnull().sum()

# making model to predict legislation based on object of search

# from sklearn.pipeline import make_pipeline
# from sklearn.preprocessing import OneHotEncoder
# train = data_filling_nans.loc[data_filling_nans['Legislation'].notna(),['Legislation','Object of search']]
# test = data_filling_nans.loc[data_filling_nans['Legislation'].isnull(),['Legislation','Object of search']]
# pipe = make_pipeline(
#   OneHotEncoder()  ,
#     GradientBoostingClassifier()
# )
#     #pickling the model
# pipe.fit(train[['Object of search']],train['Legislation'])
# import joblib
# joblib.dump(pipe, 'pickles/pipeline_legislation.pickle') 

import joblib
test = data_filling_nans.loc[data_filling_nans['Legislation'].isnull(),['Legislation','Object of search']]
pipe = joblib.load('pickles/pipeline_legislation.pickle')

pred =  pipe.predict(test[['Object of search']])
data_filling_nans.loc[data_filling_nans['Legislation'].isnull(),'Legislation'] = pred

pred

data_filling_nans.isnull().sum()

# function for getting coordinates for any place

from opencage.geocoder import OpenCageGeocode

# Replace YOUR_API_KEY with your actual API key
def getting_lat_long(address) : 
    geocoder = OpenCageGeocode("fc77af32e6754b9885e7ee8268bb1d23")

    # Address to be geocoded
    # Perform geocoding
    result = geocoder.geocode(address)

    # Print latitude and longitude
    if result and len(result):
        latitude = result[0]['geometry']['lat']
        longitude = result[0]['geometry']['lng']
#         print(f"Latitude: {latitude}, Longitude: {longitude}")
#     else:
#         print("Unable to geocode the address.")
    return {'Latitude':latitude ,'Longitude' :longitude}


# coordinates for stations that have all nulls in latitude and longitude 

# for name in ['nottinghamshire', 'south-yorkshire']:
#     print(getting_lat_long(name))
print({'Latitude': 53.1459288, 'Longitude': -1.0214971})
print({'Latitude': 53.4815333, 'Longitude': -1.3810422})

# filling the nulls 

from sklearn.impute import SimpleImputer
known_stations = {'nottinghamshire': {'Latitude': 53.1459288, 'Longitude': -1.0214971},'south-yorkshire': {'Latitude': 53.4815333, 'Longitude': -1.3810422}}
data_imputed = data_filling_nans.copy()

# Create SimpleImputer instance
imputer = SimpleImputer(strategy="most_frequent")

# Group the DataFrame by "station" and impute missing "Latitude" values in each group
failing_stations = []
for name, group in data_imputed.groupby("station")[['Latitude','Longitude']]:
    try :
        group_imputed = group.copy()
        idxs = group_imputed.index

        data_imputed.loc[idxs,['Latitude','Longitude']] = imputer.fit_transform(group_imputed)
    except ValueError :
        if name in known_stations.keys():
            coordinate = known_stations[name]
            for key , value in coordinate.items():
                imputer_fail = SimpleImputer(strategy='constant',fill_value=value)
                data_imputed.loc[idxs,[key]] = imputer_fail.fit_transform(group[[key]])
        else : 
            coordinate = getting_lat_long(name)
            print({name:coordinate})
            for key , value in coordinate.items():
                imputer_fail = SimpleImputer(strategy='constant',fill_value=value)
                data_imputed.loc[idxs,[key]] = imputer_fail.fit_transform(group[[key]])
        failing_stations.append(name)

failing_stations   

data_imputed.isnull().sum()

# making map chart to check the stations on the map 

import os 
import json
with open(os.path.join('pickles', 'boundriesOfNumeric.json')) as fh:
    Bounders = json.load(fh)

Bounders

### before imputing 

import pandas as pd
from shapely.geometry import Point
import geopandas as gpd
import matplotlib.pyplot as plt
from geopandas import GeoDataFrame

df = data_filling_nans[['Longitude', 'Latitude']]

geometry = [Point(xy) for xy in zip(df['Longitude'], df['Latitude'])]
gdf = GeoDataFrame(df, geometry=geometry)

# Bounding box coordinates for the United Kingdom
xmin, ymin = -10.854492, 49.823809
xmax, ymax = 1.877441, 60.945484

# Create a bounding box around the area of interest
bbox = (xmin, ymin, xmax, ymax)  # Define the coordinates of the bounding box

# Filter the world map to only include the area within the bounding box
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
world = world.cx[xmin:xmax, ymin:ymax]  # Crop the world map to the specified bounding box

# Plot the points within the cropped world map
fig, ax = plt.subplots(figsize=(10, 6))
world.plot(ax=ax, color='lightgray', edgecolor='gray')
gdf.plot(ax=ax, marker='o', color='red', markersize=15)

# Set the x-axis and y-axis limits to zoom in on the points
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

# Add title and labels
plt.title("Location Data in the United Kingdom")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

# Add a legend
ax.legend(["World Map", "Data Points"])

# Display the plot
plt.show()


# self and officer compare

data_filling_nans['Officer-defined ethnicity'].value_counts().sort_index()

import re

# Define the regex patterns and their corresponding new classes
pattern_map = {
    r'Asian': 'Asian',
    r'Black': 'Black',
    r'Mixed': 'mixed',
    r'Other': 'other',
    r'White': 'white',
    # Add more regex patterns and mappings as needed
}

# Define a mapping function that uses regex to match the keys
def map_category(category):
    for pattern, new_class in pattern_map.items():
        if re.match(pattern, category):
            return new_class
    # If no match is found, return a default value or handle accordingly

# Apply the mapping function to a column in your DataFrame
self_cop = data_filling_nans.copy()
self_cop['Self-defined map'] = self_cop['Self-defined ethnicity'].astype(str).apply(map_category)


self_cop['Self-defined map'].value_counts().sort_index()

df = getting_counts(self_cop, 'Officer-defined ethnicity', 'Self-defined map',normalize=True)
heading_properties = [('font-size', '18px')]
cell_properties = [('font-size', '16px'), ('text-align', 'left')]

dfstyle = [
    dict(selector="th", props=heading_properties),
    dict(selector="td", props=cell_properties)
]

styled_df = df.style.set_table_styles(dfstyle)

# Remove the "000" formatting from the values
styled_df = styled_df.format("{:.3f}")

styled_df


pd.crosstab(self_cop['Officer-defined ethnicity'],self_cop["Self-defined map"],normalize='index')

import seaborn as sns
cor_mat = df.copy()
plt.figure(figsize = (12,6)) # <-- just sets the figure size 
yticklabels = [label.capitalize() for label in cor_mat.index]
xticklabels = [label.capitalize() for label in cor_mat.columns]
sns.set(font_scale=1.2)
sns.heatmap(cor_mat, annot=True,    cmap='RdBu_r', linewidths=1, linecolor='black',yticklabels=yticklabels, xticklabels=xticklabels)
plt.savefig('pics/corr.png')
plt.show()

# checking discremenation

data_filling_nans.head(1)

## distro of removal and the Gender 

getting_counts(data_filling_nans,'Gender','Removal of more than just outer clothing')

pd.crosstab(data['Gender'],data['Removal of more than just outer clothing'])

## checking the distro for sensitive classes

data_filling_nans['Gender'].value_counts()

data_filling_nans['Officer-defined ethnicity'].value_counts().plot(kind='bar');
plt.xlabel('Age range');
plt.title('Distribution of Age range column')
plt.ylabel('Count');

## removing the Gender other 

data_filling_nans = data_filling_nans.loc[data_filling_nans['Gender']!='other']

## data counts for both outcome and outcome linked 

pd.crosstab(data_filling_nans['Outcome map'],data_filling_nans['Outcome linked to object of search'],normalize='all')*100

## the same as above but for each outcome values 

getting_counts(data_filling_nans,'Outcome map','Outcome linked to object of search')*100

getting_counts(data,'Outcome map','Outcome linked to object of search')*100

data['Outcome linked to object of search'].value_counts()

### getting stations that more than 90 is False after filling 

temp = getting_counts(data_filling_nans,'station','Outcome linked to object of search',normalize = True )
station_to_imporve_investigation = temp[temp[False]>.87].index

# ignored stations to have bad distribution of outcome liked 

station_to_imporve_investigation

data_filling_nans_improve = data_filling_nans[~data_filling_nans['station'].isin(station_to_imporve_investigation)].copy()

## making success rate metric

def get_success_rate(mask):
    """
    Verifies the success rate on a test set is above a provided minimum
    
    
    """
    
    Success_rate = mask[(mask['Outcome map']==True)&(mask['Outcome linked to object of search']==True)].shape[0]/mask.shape[0]
    
    
    
    return Success_rate


# stations = ['dorset']
# gender_classes = data_filling_nans['Gender'].unique()
# ethnicity_classes = data_filling_nans['Officer-defined ethnicity'].unique()
# for station in stations:
#     success_rates = {}
#     for gender_class in gender_classes:
#         for ethnicity_class in ethnicity_classes :
#             mask = (data_filling_nans['Gender'] == gender_class) & (data_filling_nans['station'] == station) & (data_filling_nans['Officer-defined ethnicity'] == ethnicity_class)
#             if np.sum(mask) > 30:
#                 success_rates[(gender_class,ethnicity_class)] = get_success_rate(data_filling_nans[mask])
#     print( f'Station {station} ....... ')
# if len(success_rates) > 1:    
# #             diff = np.max(list(success_rates.values())) - np.min(list(success_rates.values()))
# #             global_succes_rate[station] = np.mean(list(success_rates.values()))
#     print(success_rates)
#     print(np.max(list(success_rates.values())) ,np.min(list(success_rates.values())))
#     print(np.mean(list(success_rates.values())))
#     print('**********************************************************************************')
    

### Note : humberside have all values in outcome linked nans

# mask = (data['Gender'] == 'Male') & (data['station'] == 'humberside') & (data['Officer-defined ethnicity'] == 'White')
# data[mask]

def verify_no_discrimination(data , max_diff_tuple=0.05,max_diff_stations = .1 ,min_samples=30):
    """
    Verifies that no subdeparment has discrimination in between protected races
    """
    stations = data['station'].unique()
    gender_classes = data['Gender'].unique()
    ethnicity_classes = data['Officer-defined ethnicity'].unique()
    
    is_satisfied_tuple = True
    is_satisfied_stations = True
    problematic_stations = {}
    good_stations = {}
    ignored_stations = {}
    ignored_subgroups = []
    global_succes_rate = {}
    for station in stations:
        success_rates = {}
        for gender_class in gender_classes:
            for ethnicity_class in ethnicity_classes :
                mask = (data['Gender'] == gender_class) & (data['station'] == station) & (data['Officer-defined ethnicity'] == ethnicity_class)&(data['Outcome linked to object of search'].notnull())
                
                if np.sum(mask) > min_samples:
                    try :
                        rate = get_success_rate(data[mask])
                        if rate < .01:
                            ignored_subgroups.append((station,gender_class,ethnicity_class))
                            continue
                        success_rates[(gender_class,ethnicity_class)] = ( rate, np.sum(mask))
                    except ZeroDivisionError : 
                        print(station)
                        print(data[mask].isnull().sum())
        if len(success_rates) > 1:    
            diff = np.max(list(x[0] for x in success_rates.values())) - np.min(list(x[0] for x in success_rates.values()))
            global_succes_rate[station] = np.mean(list(x[0] for x in success_rates.values()))
            if diff > max_diff_tuple:
                is_satisfied_tuple = False
                problematic_stations[station]= (diff, success_rates)
            else:
                good_stations[station] =( diff, success_rates)
        else:
            ignored_stations[station]= (None, [])
        
    global_desparncey_score = np.max(list(global_succes_rate.values())) - np.min(list(global_succes_rate.values()))
    if global_desparncey_score > max_diff_stations:
        is_satisfied_stations = False
        
    return is_satisfied_tuple,is_satisfied_stations, problematic_stations, good_stations, global_desparncey_score,global_succes_rate,ignored_stations,ignored_subgroups



### do it for data filling nans

is_satisfied_tuple1,is_satisfied_stations1, problematic_stations1, good_stations1, global_desparncey_score1,global_succes_rate1,ignored_stations1,ignored_subgroups1 = verify_no_discrimination(
    data_filling_nans_improve,max_diff_tuple=.05,min_samples=100)

if not is_satisfied_tuple1:
    print("Requirement failed 😢")
    print("Global rates: {}".format(global_desparncey_score1))
    print("Num problematic departments: {}".format(len(problematic_stations1)))
    print("Num good departments: {}".format(len(good_stations1)))
    print("Num ignored departments: {} and it's {}".format(len(ignored_stations1),ignored_stations1))
    print("avg diff:", np.mean([p[0] for p in problematic_stations1.values()]))
else:
    print("Requirement satisfied! 🚀")

print("Departments analysed: {}".format(len(problematic_stations1) + len(good_stations1)))        

problematic_stations1.keys()

temp = {i:val for i , val in global_succes_rate1.items() if val <.17 }
temp

# stations that make global more than 0.1

sorted_data = sorted(global_succes_rate1.items(), key=lambda x: x[1], reverse=False)
stations, percentages = zip(*sorted_data)
plt.figure(figsize=(12, 6))
plt.bar(stations, percentages, width=0.5)
plt.xlabel('Stations',fontsize = 12)
plt.ylabel('Average Percentage',fontsize = 12)
plt.title('Average Percentages for Each Station')
plt.axhline(0.176, color='red', linestyle='--')
plt.axhline(max(percentages), color='blue', linestyle='--')
plt.fill([-.5, 6.5, 6.5, -.5], [0, 0, 0.176, 0.176], color='none', edgecolor='green', linestyle='-', linewidth=2, fill=False)
plt.xticks(rotation=45, ha='right',fontsize = 14)  # Set the rotation angle and alignment for the x-axis labels
plt.tight_layout()
plt.savefig('pics/global')
plt.show()

# Analysis of female and age 

temp = getting_counts(data[data['Gender']=='Female'],'station','Removal of more than just outer clothing')
temp

df = getting_counts(data,'Gender','Removal of more than just outer clothing',normalize=True)
heading_properties = [('font-size', '18px')]
cell_properties = [('font-size', '16px'), ('text-align', 'left')]

dfstyle = [
    dict(selector="th", props=heading_properties),
    dict(selector="td", props=cell_properties)
]

styled_df = df.style.set_table_styles(dfstyle)

# Remove the "000" formatting from the values
styled_df = styled_df.format("{:.3f}")

styled_df

getting_counts(data_filling_nans,'Gender','Age range')

# stations with full nulls
# ['humberside','cleveland', 'lancashire', 'metropolitan', 'north-yorkshire', 'surrey']
station_to_imporve_investigation_removal =  ['humberside','cleveland', 'north-yorkshire', 'surrey','south-yorkshire']

data_filling_nans_improve_removal = data_filling_nans[~data_filling_nans['station'].isin(station_to_imporve_investigation_removal)].copy()
data_filling_nans_improve_removal  =data_filling_nans_improve_removal[data_filling_nans_improve_removal['Age range']!='under 10']
data_filling_nans_improve_removal['Age range'].value_counts()

getting_counts(data_filling_nans_improve_removal,'Gender','Age range')

getting_counts(data,'Outcome map','Removal of more than just outer clothing')

getting_counts(data,'Removal of more than just outer clothing','Outcome map')

getting_counts(data,'Outcome map','Gender')

data_filling_nans['Age range'].unique()

print(getting_counts(data[data['Gender']=='Male'],'Age range','Removal of more than just outer clothing'))
print('--------------------------------------------------------------------------------')
print(getting_counts(data[data['Gender']=='Female'],'Age range','Removal of more than just outer clothing'))

## make matrics 

def get_success_rate1(mask):
    """
    Verifies the success rate on a test set is above a provided minimum
    
    
    """
    
    Success_rate = mask[mask['Removal of more than just outer clothing']==True].shape[0]/mask.shape[0]
    
    
    
    return Success_rate


# first
def get_remove_rate(mask,mask_total) :
    return mask[mask['Removal of more than just outer clothing']==True].shape[0]/mask_total[mask_total['Removal of more than just outer clothing']==True].shape[0]

### Build another one modified

def combined_rate(removal_rate , success_rate):
    weights = {'removal_rate': 0, 'success_rate': 1} # You can adjust the weights here
    combined_rate = weights['removal_rate'] * removal_rate + weights['success_rate'] * success_rate
    return combined_rate

def Check_removal(data ,gender, max_diff_tuple=0.05 ,min_samples=30):
    """
    Verifies that no subdeparment has discrimination in between protected races
    """
    stations = data['station'].unique()
    if gender == 'male' : 
            gender_classes = [data['Gender'].unique()[0]]
    else : 
        gender_classes = [data['Gender'].unique()[1]]
    age_classes = data['Age range'].unique()
    ethnicity_classes = data['Officer-defined ethnicity'].unique()
    is_satisfied_tuple = True
    problematic_stations = {}
    good_stations = {}
    ignored_subs = {}
    ignored_stations = {}
    for station in stations:
        success_rates = {}
        for gender_class in gender_classes:
            gender_mask = (data['Gender'] == gender_class) & (data['station'] == station) &(data['Removal of more than just outer clothing'].notnull())
            for age_class in age_classes :
                for ethnicity_class in ethnicity_classes :  
                    mask = (data['Gender'] == gender_class) & (data['Officer-defined ethnicity'] == ethnicity_class)&(data['station'] == station) & (data['Age range'] == age_class)&(data['Removal of more than just outer clothing'].notnull())
                    if np.sum(mask) > min_samples:
                        try : 
                            rate = combined_rate(get_remove_rate(data[mask],data[gender_mask]),get_success_rate1(data[mask]))
                            if rate < .01 : 
                                continue 
                                
                            success_rates[(gender_class,age_class,ethnicity_class)] = (rate, np.sum(mask))
                        except ZeroDivisionError : 
                            if not ignored_subs.get(station) : 
                                ignored_subs[station] = [(gender_class,age_class,ethnicity_class)]
                            else : 
                                ignored_subs[station].append((gender_class,age_class,ethnicity_class))
                                
        if len(success_rates) > 1:    
            diff = np.max(list(x[0] for x in success_rates.values())) - np.min(list(x[0] for x in success_rates.values()))
#             diff2 = np.max(list(success_rates.values())) - sorted(success_rates.values())[1]
            
            if diff > max_diff_tuple :
#                 if   diff2 > max_diff_tuple : 
                is_satisfied_tuple = False
                problematic_stations[station]  = ( diff, success_rates)
#                 else  :
#                     good_stations.append((station, diff2, success_rates))
            else:
                good_stations[station]  = ( diff, success_rates)
        else:
            ignored_stations[station]  = ( None, [])
    
        
    return is_satisfied_tuple, problematic_stations, good_stations,ignored_stations ,ignored_subs



is_satisfied_tuple2, problematic_stations2, good_stations2,ignored_stations2 , igonred_subgroups2 = Check_removal(data_filling_nans,gender='female',min_samples=25,max_diff_tuple = .08)
# play_tune()

if not is_satisfied_tuple2:
    print("Requirement failed 😢")
    print("Num problematic departments: {}".format(len(problematic_stations2)))
    print("Num ignored departments: {} and it's {}".format(len(ignored_stations2),ignored_stations2))
    print("Num good departments: {}".format(len(good_stations2)))

    print("avg diff:", np.mean([p[0] for p in problematic_stations2.values()]))
else:
    print("Requirement satisfied! 🚀")

print("Departments analysed: {}".format(len(problematic_stations2) + len(good_stations2)))        